In [1]:
#!/usr/bin/env python3
"""
TS-SatFire Burned Area (BA) Label Audit
========================================
Scans every fire in the dataset for BA label quality (band 8, 1-indexed = day_arr[7]).
Produces exclusion lists for train/val/test splits.

Run on Kaggle with the TS-SatFire dataset attached.
Expected runtime: ~5-10 minutes (I/O bound, reads band headers only).
"""

# ===========================================================================
# Cell 1: Imports and setup
# ===========================================================================
import os
import glob
import json
import time
import warnings
from collections import defaultdict

import numpy as np

warnings.filterwarnings("ignore")

# Try rasterio first (preferred), fall back to tifffile
try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    import tifffile

print("=" * 70)
print("TS-SatFire Burned Area (BA) Label Audit")
print("=" * 70)

# ===========================================================================
# Cell 2: Configuration -- paths, splits, constants
# ===========================================================================

# Dataset root -- double-nested on Kaggle
DATASET_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/"
if not os.path.isdir(DATASET_ROOT):
    # Fallback for local testing
    DATASET_ROOT = "/kaggle/input/ts-satfire/ts-satfire/"
if not os.path.isdir(DATASET_ROOT):
    raise FileNotFoundError(
        f"Dataset not found. Tried:\n"
        f"  /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/\n"
        f"  /kaggle/input/ts-satfire/ts-satfire/\n"
        f"Check your Kaggle dataset attachment."
    )

print(f"Dataset root: {DATASET_ROOT}")

# Paper's validation IDs (from codebase, same for AF and BA)
VAL_IDS = [
    "20568194", "20701026", "20562846", "20700973", "24462610",
    "24462788", "24462753", "24103571", "21998313", "21751303",
    "22141596", "21999381", "22712904",
]

# Paper's AF test fires (17 named fires, global)
AF_TEST_FIRES = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire",
    "sparks_lake_fire", "lytton_fire", "chuckegg_creek_fire",
    "swedish_fire", "sydney_fire", "thomas_fire", "tubbs_fire",
    "carr_fire", "camp_fire", "creek_fire", "blue_ridge_fire",
    "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]

# BA label is in band 8 (1-indexed) = index 7 (0-indexed)
BA_BAND_INDEX = 7  # 0-indexed

# AF label band for reference comparison
AF_BAND_INDEX = 6  # 0-indexed

# Known AF no-label fire IDs (from our previous audit)
AF_NO_LABEL_IDS = [
    "20777207", "20777386", "21693566", "21751305", "21751309",
    "21889672", "21889683", "21889697", "21889719", "21889734",
    "21889754", "21890056", "21997775", "22712904", "22712973",
    "22713339", "23036871", "23860939", "23860978", "23861018",
    "23861131", "24332700",
]


# ===========================================================================
# Cell 3: Helper functions
# ===========================================================================

def read_geotiff_array(path):
    """Read a GeoTIFF into a numpy array (C, H, W)."""
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)
    else:
        arr = tifffile.imread(path).astype(np.float32)
        if arr.ndim == 2:
            arr = arr[np.newaxis]
        return arr


def check_ba_label(day_path):
    """
    Check BA label quality for a single VIIRS_Day GeoTIFF.
    
    Returns dict with:
        - has_band: whether band 8 exists
        - total_pixels: total pixel count
        - nan_pixels: NaN pixel count
        - finite_pixels: non-NaN pixel count
        - burned_pixels: pixels with burned area signal
        - unique_values: sample of unique finite values (for understanding encoding)
        - is_usable: True if the band has meaningful (non-all-NaN) data
    """
    result = {
        "path": os.path.basename(day_path),
        "has_band": False,
        "total_pixels": 0,
        "nan_pixels": 0,
        "finite_pixels": 0,
        "burned_pixels": 0,
        "fire_frac": 0.0,
        "unique_finite_sample": [],
        "is_usable": False,
    }
    
    try:
        arr = read_geotiff_array(day_path)
    except Exception as e:
        result["error"] = str(e)
        return result
    
    n_bands = arr.shape[0]
    
    if n_bands <= BA_BAND_INDEX:
        result["has_band"] = False
        return result
    
    result["has_band"] = True
    ba_band = arr[BA_BAND_INDEX]
    
    total = ba_band.size
    nan_count = int(np.isnan(ba_band).sum())
    finite_count = total - nan_count
    
    result["total_pixels"] = total
    result["nan_pixels"] = nan_count
    result["finite_pixels"] = finite_count
    
    if finite_count > 0:
        finite_vals = ba_band[~np.isnan(ba_band)]
        # For BA, we need to figure out the encoding
        # The paper says labels are union of accumulated AF + NIFC perimeters
        # Likely: 0 = no burn, 1 = burned, or similar binary/threshold scheme
        # We also check for the same pattern as AF (values >= 7)
        # But first, let us just record what values exist
        unique_vals = np.unique(finite_vals)
        if len(unique_vals) <= 20:
            result["unique_finite_sample"] = unique_vals.tolist()
        else:
            result["unique_finite_sample"] = [
                float(unique_vals.min()),
                float(np.median(unique_vals)),
                float(unique_vals.max()),
                f"({len(unique_vals)} unique values)",
            ]
        
        # Count burned pixels: try multiple thresholds to understand encoding
        # Binary: value == 1
        result["burned_binary"] = int((finite_vals == 1).sum())
        # Threshold >= 1 (any positive)
        result["burned_ge1"] = int((finite_vals >= 1).sum())
        # Threshold > 0
        result["burned_gt0"] = int((finite_vals > 0).sum())
        # For consistency with AF pattern (>= 7)
        result["burned_ge7"] = int((finite_vals >= 7).sum())
        
        # Use the most likely encoding: > 0 means burned
        result["burned_pixels"] = int((finite_vals > 0).sum())
        result["fire_frac"] = result["burned_pixels"] / total
        
        result["is_usable"] = True
    else:
        result["is_usable"] = False
    
    return result


def classify_fire(fire_name, val_ids, af_test_fires):
    """Classify a fire folder into train/val/af_test/ba_test_candidate."""
    if fire_name in af_test_fires:
        return "af_test"
    if fire_name in val_ids:
        return "val"
    # Check if numeric (train fires are numeric IDs from 2017-2020)
    if fire_name.isdigit():
        return "train"
    # Named but not in AF test -- could be BA test or other
    return "other_named"


# ===========================================================================
# Cell 4: Scan all fires
# ===========================================================================

print("\nScanning all fire directories...")
t0 = time.time()

all_fires = sorted([
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
])
print(f"Found {len(all_fires)} fire directories")

# Classify each fire
fire_categories = {}
for f in all_fires:
    cat = classify_fire(f, VAL_IDS, AF_TEST_FIRES)
    fire_categories[f] = cat

cat_counts = defaultdict(int)
for cat in fire_categories.values():
    cat_counts[cat] += 1
print(f"\nFire categories:")
for cat, cnt in sorted(cat_counts.items()):
    print(f"  {cat}: {cnt}")


# ===========================================================================
# Cell 5: Audit BA labels for every fire
# ===========================================================================

print("\n" + "=" * 70)
print("Auditing BA labels (band 8) for all fires...")
print("=" * 70)

fire_audit = {}

for idx, fire_name in enumerate(all_fires):
    fire_dir = os.path.join(DATASET_ROOT, fire_name)
    day_dir = os.path.join(fire_dir, "VIIRS_Day")
    
    fire_info = {
        "category": fire_categories[fire_name],
        "has_day_dir": os.path.isdir(day_dir),
        "n_day_files": 0,
        "n_usable_ba": 0,
        "n_all_nan_ba": 0,
        "n_no_band": 0,
        "n_with_burned_pixels": 0,
        "total_burned_pixels": 0,
        "total_pixels_checked": 0,
        "day_results": [],
        "value_samples": [],
    }
    
    if not fire_info["has_day_dir"]:
        fire_audit[fire_name] = fire_info
        continue
    
    day_files = sorted(glob.glob(os.path.join(day_dir, "*.tif")))
    fire_info["n_day_files"] = len(day_files)
    
    for dp in day_files:
        res = check_ba_label(dp)
        fire_info["day_results"].append(res)
        
        if not res["has_band"]:
            fire_info["n_no_band"] += 1
        elif res["is_usable"]:
            fire_info["n_usable_ba"] += 1
            fire_info["total_burned_pixels"] += res["burned_pixels"]
            fire_info["total_pixels_checked"] += res["total_pixels"]
            if res["burned_pixels"] > 0:
                fire_info["n_with_burned_pixels"] += 1
            if res["unique_finite_sample"]:
                fire_info["value_samples"].extend(
                    [v for v in res["unique_finite_sample"] if isinstance(v, (int, float))]
                )
        else:
            fire_info["n_all_nan_ba"] += 1
    
    fire_audit[fire_name] = fire_info
    
    if (idx + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f"  Processed {idx+1}/{len(all_fires)} fires ({elapsed:.1f}s)")

elapsed = time.time() - t0
print(f"\nAudit complete in {elapsed:.1f}s")


# ===========================================================================
# Cell 6: Analyze BA label encoding (what values mean "burned"?)
# ===========================================================================

print("\n" + "=" * 70)
print("BA Label Encoding Analysis")
print("=" * 70)

# Collect all unique value samples across fires
all_value_samples = set()
for fire_name, info in fire_audit.items():
    for v in info["value_samples"]:
        if isinstance(v, (int, float)) and not np.isnan(v):
            all_value_samples.add(v)

if all_value_samples:
    sorted_vals = sorted(all_value_samples)
    print(f"\nAll unique finite BA band values seen across dataset:")
    if len(sorted_vals) <= 50:
        print(f"  {sorted_vals}")
    else:
        print(f"  Min: {sorted_vals[0]}, Max: {sorted_vals[-1]}")
        print(f"  Count: {len(sorted_vals)} unique values")
        print(f"  First 20: {sorted_vals[:20]}")
        print(f"  Last 10: {sorted_vals[-10:]}")

# Check encoding patterns from a subset of fires with usable data
print("\nEncoding check (burned pixel counts by threshold):")
encoding_check = {"binary_eq1": 0, "ge1": 0, "gt0": 0, "ge7": 0, "total_finite": 0}
n_checked = 0
for fire_name, info in fire_audit.items():
    for res in info["day_results"]:
        if res.get("is_usable", False):
            encoding_check["binary_eq1"] += res.get("burned_binary", 0)
            encoding_check["ge1"] += res.get("burned_ge1", 0)
            encoding_check["gt0"] += res.get("burned_gt0", 0)
            encoding_check["ge7"] += res.get("burned_ge7", 0)
            encoding_check["total_finite"] += res["finite_pixels"]
            n_checked += 1

if n_checked > 0:
    print(f"  Across {n_checked} usable day files:")
    for key, val in encoding_check.items():
        pct = val / encoding_check["total_finite"] * 100 if encoding_check["total_finite"] > 0 else 0
        print(f"    {key}: {val:,} pixels ({pct:.4f}%)")


# ===========================================================================
# Cell 7: Produce per-category summary and exclusion lists
# ===========================================================================

print("\n" + "=" * 70)
print("BA Label Quality Summary by Split")
print("=" * 70)

CATEGORIES = ["train", "val", "af_test", "other_named"]
CAT_LABELS = {
    "train": "TRAIN (numeric, non-val, 2017-2020)",
    "val": "VALIDATION (paper's 13 VAL_IDS)",
    "af_test": "AF TEST (17 named fires -- not BA test!)",
    "other_named": "OTHER NAMED (likely BA/Pred test from 2021)",
}

ba_exclude = {"train": [], "val": [], "af_test": [], "other_named": []}

for cat in CATEGORIES:
    fires_in_cat = [f for f, info in fire_audit.items() if info["category"] == cat]
    if not fires_in_cat:
        continue
    
    print(f"\n--- {CAT_LABELS.get(cat, cat)} ({len(fires_in_cat)} fires) ---")
    
    # Sub-categorize
    no_day_dir = []
    no_day_files = []
    zero_usable = []
    mostly_nan = []
    partial = []
    full = []
    
    for f in fires_in_cat:
        info = fire_audit[f]
        if not info["has_day_dir"]:
            no_day_dir.append(f)
        elif info["n_day_files"] == 0:
            no_day_files.append(f)
        elif info["n_usable_ba"] == 0:
            zero_usable.append(f)
        else:
            usable_frac = info["n_usable_ba"] / info["n_day_files"]
            if usable_frac < 0.5:
                mostly_nan.append(f)
            elif usable_frac < 1.0:
                partial.append(f)
            else:
                full.append(f)
    
    print(f"  FULL labels (100% days usable):     {len(full)}")
    print(f"  PARTIAL labels (50-99% usable):     {len(partial)}")
    print(f"  MOSTLY NaN (<50% usable):           {len(mostly_nan)}")
    print(f"  ZERO usable BA labels:              {len(zero_usable)}")
    print(f"  No VIIRS_Day files:                 {len(no_day_files)}")
    print(f"  No VIIRS_Day directory:             {len(no_day_dir)}")
    
    # Fires with zero burned pixels even among usable days
    zero_burned = []
    for f in fires_in_cat:
        info = fire_audit[f]
        if info["n_usable_ba"] > 0 and info["n_with_burned_pixels"] == 0:
            zero_burned.append(f)
    if zero_burned:
        print(f"  Has labels but ZERO burned pixels:  {len(zero_burned)}")
    
    # Build exclusion list: fires that should NOT be used for BA training/eval
    # Exclude: no day dir, no day files, zero usable, mostly NaN
    exclude = no_day_dir + no_day_files + zero_usable + mostly_nan
    ba_exclude[cat] = exclude
    
    if exclude:
        print(f"\n  >> EXCLUDE from BA ({len(exclude)} fires):")
        for f in sorted(exclude):
            info = fire_audit[f]
            reason = ""
            if not info["has_day_dir"]:
                reason = "no VIIRS_Day dir"
            elif info["n_day_files"] == 0:
                reason = "0 day files"
            elif info["n_usable_ba"] == 0:
                reason = f"0/{info['n_day_files']} days usable (all NaN)"
            else:
                reason = f"{info['n_usable_ba']}/{info['n_day_files']} usable (<50%)"
            print(f"     {f}: {reason}")
    
    # Print detail for partial fires
    if partial and cat in ("val", "af_test", "other_named"):
        print(f"\n  Partial fires (usable but some missing days):")
        for f in sorted(partial):
            info = fire_audit[f]
            frac = info["n_usable_ba"] / info["n_day_files"]
            burned_frac = info["total_burned_pixels"] / info["total_pixels_checked"] if info["total_pixels_checked"] > 0 else 0
            print(f"     {f}: {info['n_usable_ba']}/{info['n_day_files']} days usable ({frac:.0%}), "
                  f"burn fraction {burned_frac:.5f}")


# ===========================================================================
# Cell 8: Identify BA test fires (2021 fires)
# ===========================================================================

print("\n" + "=" * 70)
print("Identifying BA/Pred Test Fires (2021 fires)")
print("=" * 70)
print("\nThe paper says BA test set = 24 fire events from 2021.")
print("These should be in the 'other_named' category or identifiable by ID patterns.")

# The 2021 fires for BA/Pred test are typically identified by their GlobFire
# or MODIS burned area product IDs. Let's check what's in 'other_named' 
# and also look for any non-AF-test, non-val, non-train fires.

other_named = [f for f, info in fire_audit.items() if info["category"] == "other_named"]
print(f"\n'other_named' fires ({len(other_named)}):")
for f in sorted(other_named):
    info = fire_audit[f]
    ba_status = f"{info['n_usable_ba']}/{info['n_day_files']} usable"
    burned = f"burned_frac={info['total_burned_pixels']/info['total_pixels_checked']:.5f}" if info['total_pixels_checked'] > 0 else "no data"
    print(f"  {f}: {ba_status}, {burned}")

# Also check: are there numeric fires that might be 2021?
# The paper uses GlobFire IDs for 2017-2020 and MODIS product IDs for 2021
# We can try to identify 2021 fires by checking if they have a specific pattern
# or by looking at the ROI CSV files if they exist in the dataset
roi_dir = os.path.join(DATASET_ROOT, "roi")
if os.path.isdir(roi_dir):
    print(f"\nROI directory found! Contents:")
    for f in sorted(os.listdir(roi_dir)):
        print(f"  {f}")
else:
    print("\nNo 'roi' directory found in dataset root.")
    # Check one level up
    parent = os.path.dirname(DATASET_ROOT.rstrip("/"))
    roi_dir2 = os.path.join(parent, "roi")
    if os.path.isdir(roi_dir2):
        print(f"Found roi at: {roi_dir2}")
        for f in sorted(os.listdir(roi_dir2)):
            print(f"  {f}")


# ===========================================================================
# Cell 9: Generate Python exclusion lists for copy-paste into training code
# ===========================================================================

print("\n" + "=" * 70)
print("COPY-PASTE EXCLUSION LISTS FOR BA TRAINING")
print("=" * 70)

print("\n# --- BA Train Exclusion List ---")
train_exclude = sorted(ba_exclude["train"])
print(f"BA_TRAIN_EXCLUDE = [")
for i in range(0, len(train_exclude), 5):
    chunk = train_exclude[i:i+5]
    line = ", ".join(f'"{x}"' for x in chunk)
    print(f"    {line},")
print(f"]")
print(f"# Total: {len(train_exclude)} fires excluded from BA training")

print(f"\n# --- BA Val Exclusion List ---")
val_exclude = sorted(ba_exclude["val"])
print(f"BA_VAL_EXCLUDE = [")
for x in val_exclude:
    print(f'    "{x}",')
print(f"]")
print(f"# Total: {len(val_exclude)} fires excluded from BA validation")

print(f"\n# --- BA AF-Test Exclusion List ---")
aftest_exclude = sorted(ba_exclude["af_test"])
print(f"BA_AFTEST_EXCLUDE = [")
for x in aftest_exclude:
    print(f'    "{x}",')
print(f"]")
print(f"# Total: {len(aftest_exclude)} fires excluded from AF test (BA eval)")

print(f"\n# --- Other Named Exclusion List ---")
other_exclude = sorted(ba_exclude["other_named"])
print(f"BA_OTHER_EXCLUDE = [")
for x in other_exclude:
    print(f'    "{x}",')
print(f"]")
print(f"# Total: {len(other_exclude)} fires excluded from other named fires")


# ===========================================================================
# Cell 10: Detailed per-fire report (for reference)
# ===========================================================================

print("\n" + "=" * 70)
print("Detailed Per-Fire BA Audit (all fires with issues)")
print("=" * 70)

for cat in CATEGORIES:
    fires_in_cat = sorted([f for f, info in fire_audit.items() if info["category"] == cat])
    if not fires_in_cat:
        continue
    
    print(f"\n=== {CAT_LABELS.get(cat, cat)} ===")
    for f in fires_in_cat:
        info = fire_audit[f]
        
        # Only print fires with issues or test/val fires
        has_issue = (
            not info["has_day_dir"]
            or info["n_day_files"] == 0
            or info["n_usable_ba"] == 0
            or (info["n_day_files"] > 0 and info["n_usable_ba"] < info["n_day_files"])
        )
        
        if has_issue or cat in ("val", "af_test", "other_named"):
            if not info["has_day_dir"]:
                status = "NO_DAY_DIR"
            elif info["n_day_files"] == 0:
                status = "NO_FILES"
            elif info["n_usable_ba"] == 0:
                status = "ALL_NAN"
            elif info["n_usable_ba"] < info["n_day_files"]:
                frac = info["n_usable_ba"] / info["n_day_files"]
                status = f"PARTIAL({frac:.0%})"
            else:
                status = "FULL"
            
            burned_info = ""
            if info["total_pixels_checked"] > 0:
                bf = info["total_burned_pixels"] / info["total_pixels_checked"]
                burned_info = f" burn_frac={bf:.6f}"
            
            print(f"  {f}: {status} ({info['n_usable_ba']}/{info['n_day_files']} days){burned_info}")


# ===========================================================================
# Cell 11: Compare with AF exclusion list
# ===========================================================================

print("\n" + "=" * 70)
print("Cross-Reference: BA vs AF Exclusion Lists")
print("=" * 70)

ba_all_exclude = set()
for cat_list in ba_exclude.values():
    ba_all_exclude.update(cat_list)

af_set = set(AF_NO_LABEL_IDS)
ba_set = ba_all_exclude

both = af_set & ba_set
af_only = af_set - ba_set
ba_only = ba_set - af_set

print(f"\nExcluded in BOTH AF and BA:  {len(both)}")
print(f"Excluded in AF only:         {len(af_only)}")
print(f"Excluded in BA only:         {len(ba_only)}")

if af_only:
    print(f"\nAF-only exclusions (have BA labels but not AF):")
    for f in sorted(af_only):
        print(f"  {f}")

if ba_only:
    print(f"\nBA-only exclusions (have AF labels but not BA):")
    for f in sorted(ba_only):
        print(f"  {f}")


# ===========================================================================
# Cell 12: Summary statistics
# ===========================================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

total_fires = len(all_fires)
total_with_day = sum(1 for info in fire_audit.values() if info["has_day_dir"] and info["n_day_files"] > 0)
total_with_ba = sum(1 for info in fire_audit.values() if info["n_usable_ba"] > 0)
total_train_usable = sum(
    1 for f, info in fire_audit.items()
    if info["category"] == "train" and info["n_usable_ba"] > 0 and f not in ba_exclude["train"]
)
total_val_usable = sum(
    1 for f, info in fire_audit.items()
    if info["category"] == "val" and info["n_usable_ba"] > 0 and f not in ba_exclude["val"]
)

print(f"Total fires in dataset:           {total_fires}")
print(f"Fires with VIIRS_Day data:        {total_with_day}")
print(f"Fires with any usable BA label:   {total_with_ba}")
print(f"Train fires usable for BA:        {total_train_usable}")
print(f"Val fires usable for BA:          {total_val_usable}")
print(f"BA train exclusions:              {len(ba_exclude['train'])}")
print(f"BA val exclusions:                {len(ba_exclude['val'])}")

# Count total usable windows for training
total_train_windows = 0
for f, info in fire_audit.items():
    if info["category"] == "train" and f not in ba_exclude["train"]:
        # For TS=2, windows = n_usable_days - 1
        if info["n_usable_ba"] >= 2:
            total_train_windows += info["n_usable_ba"] - 1
        elif info["n_usable_ba"] == 1:
            # Single day -- can't make a 2-day window unless we pad
            pass

print(f"\nEstimated BA train windows (TS=2): {total_train_windows}")
print(f"(Each window = 2 consecutive days with usable BA labels)")

total_elapsed = time.time() - t0
print(f"\nTotal audit time: {total_elapsed:.1f}s")
print("\nDone! Use the exclusion lists above when building the BA training pipeline.")

TS-SatFire Burned Area (BA) Label Audit
Dataset root: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/

Scanning all fire directories...
Found 192 fire directories

Fire categories:
  af_test: 17
  other_named: 24
  train: 138
  val: 13

Auditing BA labels (band 8) for all fires...
  Processed 20/192 fires (48.3s)
  Processed 40/192 fires (101.0s)
  Processed 60/192 fires (138.2s)
  Processed 80/192 fires (188.7s)
  Processed 100/192 fires (221.3s)
  Processed 120/192 fires (272.5s)
  Processed 140/192 fires (333.7s)
  Processed 160/192 fires (394.9s)
  Processed 180/192 fires (497.1s)

Audit complete in 513.5s

BA Label Encoding Analysis

All unique finite BA band values seen across dataset:
  Min: 2119.0, Max: 908660.0
  Count: 11660 unique values
  First 20: [2119.0, 2126.0, 2376.0, 2816.0, 2937.0, 3020.0, 3197.0, 3206.0, 3327.0, 3365.0, 3366.0, 3422.0, 3537.0, 3558.0, 3590.0, 3615.0, 3616.0, 3790.0, 3808.0, 7622.0]
  Last 10: [906648.0, 906659.0, 907054.0, 907055.0, 907056.0